# 06 · SQL recap: practice questions and answers
Answers from the recap session and final quiz, in their **corrected** form, grouped by topic.
Cells without saved output were corrected after review: re-run them to see results.

> **Run one notebook at a time.** DuckDB allows only one open connection to `practice.duckdb`. Before opening another notebook, use *Kernel → Shut Down Kernel* in this one.

In [ ]:
import duckdb

# Connect to the local DuckDB file built in 00_setup_and_load
con = duckdb.connect('../data/practice.duckdb')

# Memory settings: needed for the 59M-row sales_daily table on a laptop
con.sql("SET memory_limit = '4GB'")
con.sql("SET preserve_insertion_order = false")
con.sql("SET temp_directory = '../data/tmp'")

## GROUP BY
**Q:** Which stores drove the most revenue in calendar year 2015? Highest first.

In [ ]:
con.sql("""
SELECT store_id,
       SUM(revenue) AS tot_revenue
FROM sales_daily
WHERE date >= '2015-01-01' AND date < '2016-01-01'   -- safest year filter
GROUP BY store_id
ORDER BY tot_revenue DESC
""").df()

,store_id,tot_revenue
0,CA_3,7.054106e+06
1,CA_1,5.058087e+06
2,TX_3,4.395219e+06
3,WI_2,4.298561e+06
4,TX_2,4.290981e+06
5,CA_2,3.906888e+06
6,WI_1,3.789069e+06
7,TX_1,3.535360e+06
8,WI_3,3.305906e+06
9,CA_4,2.782281e+06


**Q:** For store CA_1, which items sold more than 10,000 units in total? Highest first.

In [ ]:
con.sql("""
SELECT item_id,
       SUM(units) AS tot_unit_sold
FROM sales_daily
WHERE store_id = 'CA_1'                 -- filter rows before grouping
GROUP BY item_id
HAVING SUM(units) > 10000               -- filter groups after grouping (portable form, no alias)
ORDER BY tot_unit_sold DESC
""").df()

,item_id,tot_unit_sold
0,FOODS_3_090,128855.0
1,FOODS_3_586,88846.0
2,FOODS_3_252,76055.0
3,FOODS_3_120,63107.0
4,FOODS_3_714,54617.0
...,...,...
137,FOODS_3_594,10231.0
138,FOODS_3_474,10212.0
139,FOODS_3_688,10168.0
140,HOBBIES_1_067,10067.0


## Window functions
**Q:** Each store's total revenue and its percentage share of all stores' revenue. Highest first.

In [ ]:
# Window on top of an aggregate: inner SUM = store total, outer SUM OVER () = all stores
con.sql("""
SELECT store_id,
       SUM(revenue) AS tot_revenue,
       ROUND(SUM(revenue) * 100.0 / SUM(SUM(revenue)) OVER (), 2) AS pct_revenue
FROM sales_daily
GROUP BY store_id
ORDER BY pct_revenue DESC
""").df()

**Q:** For each store, its 3 best weeks by revenue.

In [ ]:
# CTE needed: a window result (rank) cannot be filtered in WHERE
con.sql("""
WITH store_week_rev AS (
    SELECT store_id,
           wm_yr_wk,
           SUM(revenue) AS tot_weekly_rev,
           RANK() OVER (PARTITION BY store_id ORDER BY SUM(revenue) DESC) AS rank_wk
    FROM sales_daily
    GROUP BY store_id, wm_yr_wk
)
SELECT store_id, wm_yr_wk, tot_weekly_rev, rank_wk
FROM store_week_rev
WHERE rank_wk <= 3
ORDER BY store_id, rank_wk
""").df()

## JOINs
**Q1a:** Every item with its ABC class and XYZ class side by side.

In [ ]:
# LEFT JOIN from item_abc keeps every item, even one missing from item_xyz
con.sql("""
SELECT a.item_id,
       a.abc_rev,
       x.xyz_items
FROM item_abc a
LEFT JOIN item_xyz x
       ON a.item_id = x.item_id
""").df()

**Q1b:** Items that have an ABC class but are missing from the XYZ table (anti-join).

In [ ]:
# Test the right table's KEY for NULL: it is only NULL when there is no match
con.sql("""
SELECT a.item_id,
       a.abc_rev
FROM item_abc a
LEFT JOIN item_xyz x
       ON a.item_id = x.item_id
WHERE x.item_id IS NULL
""").df()

**Result:** no rows: every ABC item also has an XYZ class.

**Q2:** Prove the row count of a join between `sales_daily` and `calendar` on `wm_yr_wk` only.

In [ ]:
# Wrong join: calendar has 7 rows per week, so each daily sales row matches 7 days
con.sql("""
SELECT COUNT(*) AS join_table_rows
FROM sales_daily s
INNER JOIN calendar c
        ON s.wm_yr_wk = c.wm_yr_wk
""").df()

,join_table_rows
0,414267630


In [ ]:
con.sql("SELECT COUNT(*) AS sales_daily_row_count FROM sales_daily").df()

,sales_daily_row_count
0,59181090


**Result:** 414,267,630 vs 59,181,090 rows = **7× fan-out**. `wm_yr_wk` is not unique in `calendar` (one row per day).
Fix: join on the daily key (`date` or `d`). Any SUM over the wrong join would be 7× too high.

## Recurring mistakes: spot-the-errors exercise
**Q:** For store CA_1, each item's share of 2015 revenue (%), highest first; flag items above 1% as 'Top'.

In [ ]:
con.sql("""
WITH item_rev AS (
    SELECT item_id,
           SUM(revenue) AS rev
    FROM sales_daily
    WHERE store_id = 'CA_1'
      AND date >= '2015-01-01' AND date < '2016-01-01'
    GROUP BY item_id
),
item_pct AS (
    SELECT item_id,
           rev * 100.0 / SUM(rev) OVER () AS pct_raw     -- 100.0, multiply before dividing
    FROM item_rev
)
SELECT item_id,
       ROUND(pct_raw, 2) AS pct,                          -- round for display only
       CASE WHEN pct_raw > 1 THEN 'Top' ELSE 'Other' END AS flag   -- classify on unrounded value
FROM item_pct
ORDER BY pct_raw DESC
""").df()

## Final quiz
**Q1.** Save a table `store_item_coverage`: per store, distinct items that sold at least one unit in 2016, and the % of the full catalog. Lowest first.

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE store_item_coverage AS
SELECT store_id,
       COUNT(DISTINCT item_id) AS items_sold_2016,
       COUNT(DISTINCT item_id) * 100.0
         / (SELECT COUNT(DISTINCT item_id) FROM sales_daily) AS coverage_pct   -- catalog size calculated, not hardcoded
FROM sales_daily
WHERE date >= '2016-01-01' AND date < '2017-01-01'
  AND units > 0
GROUP BY store_id
""")

In [ ]:
# Sort when reading the table, not when saving it
con.sql("SELECT * FROM store_item_coverage ORDER BY coverage_pct").df()

**Q2.** FOODS_3_090 in CA_1: weekly units and that week's sell price from the price table, latest week first.

In [ ]:
con.sql("""
WITH weekly AS (
    -- aggregate to item × store × week first, so both sides of the join have the same grain
    SELECT item_id, store_id, wm_yr_wk,
           SUM(units) AS tot_units
    FROM sales_daily
    WHERE item_id = 'FOODS_3_090'
      AND store_id = 'CA_1'
    GROUP BY item_id, store_id, wm_yr_wk
)
SELECT w.wm_yr_wk,
       w.tot_units,
       p.sell_price
FROM weekly w
LEFT JOIN prices p
       ON  w.item_id  = p.item_id           -- prices grain = item × store × week → all three keys
       AND w.store_id = p.store_id
       AND w.wm_yr_wk = p.wm_yr_wk
ORDER BY w.wm_yr_wk DESC
""").df()

**Q3.** CA_3: the average absolute week-over-week change in total store units (one number).

In [ ]:
con.sql("""
WITH weekly AS (
    SELECT wm_yr_wk,
           SUM(units) AS tot_units,
           ABS(SUM(units) - LAG(SUM(units)) OVER (ORDER BY wm_yr_wk)) AS abs_wow_change
    FROM sales_daily
    WHERE store_id = 'CA_3'
    GROUP BY wm_yr_wk
)
-- plain aggregate (no OVER) → one row; AVG skips the NULL from the first week
SELECT AVG(abs_wow_change) AS avg_abs_wow_change
FROM weekly
""").df()

**Q4.** WI_1: each week's revenue in 2015 and the running total since the start of the year.

In [ ]:
con.sql("""
WITH weekly_rev AS (
    SELECT store_id,
           wm_yr_wk,
           SUM(revenue) AS rev_week
    FROM sales_daily
    WHERE store_id = 'WI_1'
      AND date BETWEEN '2015-01-01' AND '2015-12-31'
    GROUP BY store_id, wm_yr_wk
)
SELECT store_id,
       wm_yr_wk,
       rev_week,
       SUM(rev_week) OVER (ORDER BY wm_yr_wk
                           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_rev
FROM weekly_rev
ORDER BY wm_yr_wk
""").df()

,store_id,wm_yr_wk,rev_week,cum_rev
0,WI_1,11448,20596.24,20596.24
1,WI_1,11449,63422.72,84018.96
2,WI_1,11450,76442.29,160461.25
3,WI_1,11451,71285.54,231746.79
4,WI_1,11452,69727.67,301474.46
5,WI_1,11501,74597.54,376072.00
6,WI_1,11502,80299.22,456371.22
7,WI_1,11503,74105.83,530477.05
8,WI_1,11504,74220.16,604697.21
9,WI_1,11505,78105.65,682802.86


**Q5.** Each store's top 5 items by units sold in 2015, with each item's share of the store's 2015 units.

In [ ]:
# Share is calculated BEFORE filtering to the top 5, so it is the share of the whole store
con.sql("""
WITH store_item_sale AS (
    SELECT store_id,
           item_id,
           SUM(units) AS units_sold_2015,
           SUM(units) * 100.0 / SUM(SUM(units)) OVER (PARTITION BY store_id) AS item_perc_share,
           ROW_NUMBER() OVER (PARTITION BY store_id ORDER BY SUM(units) DESC) AS item_sale_rank
    FROM sales_daily
    WHERE date BETWEEN '2015-01-01' AND '2015-12-31'
    GROUP BY store_id, item_id
)
SELECT store_id,
       item_id,
       units_sold_2015,
       ROUND(item_perc_share, 2) AS item_pct_share,
       item_sale_rank
FROM store_item_sale
WHERE item_sale_rank <= 5
ORDER BY store_id, item_sale_rank        -- rank 1 = highest, so ascending
""").df()